# HydroCNHS with reservoir decisions

## Goal
Keep HydroCNHS hydrology, routing, calibration tools and its four coupling APIs. This extension supplies decisions and common reservoir physics through the Dam API. The LLM proposes a release; deterministic code applies it.

![Architecture](../docs/figures/hydrocnhs-coupling-framework-en.png)

## Setup
Python 3.10+ with repository dependencies. All cells use local files; no provider calls.

In [1]:
from pathlib import Path
import json, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/hydrocnhs').is_dir())
sys.path.insert(0, str(ROOT/'src'))
from llm_reservoir_agent.workflow import load_settings, run_settings, verify_settings_output
from llm_reservoir_agent.verification import verify_run
from unittest.mock import patch
def ensure_offline(config):
    settings = load_settings(ROOT/config)
    assert settings.report['planned_llm_calls'] == 0
    output = Path(settings.report['output'])
    if not output.exists():
        with patch('urllib.request.urlopen', side_effect=AssertionError('network forbidden')):
            run_settings(settings)
    return settings, verify_settings_output(settings)
print('Checkout:', ROOT.name)


Checkout: clean-upgrade-99ea43e42a


## Steps

### Inspect bundled capabilities

In [2]:
import hydrocnhs
source=json.loads((ROOT/'third_party/hydrocnhs/SOURCE.json').read_text())
print('Native HydroCNHS commit:', source['commit'])
print('Rainfall-runoff:', sorted(p.name for p in (ROOT/'src/hydrocnhs/rainfall_runoff_model').glob('*.py')))
print('Methods: rule-based benchmark, zero-shot, DKP')

Native HydroCNHS commit: fdd5097a3fcabb312c6c3739efc71da2d67c47da
Rainfall-runoff: ['__init__.py', 'abcd.py', 'gwlf.py', 'pet_hamon.py']
Methods: rule-based benchmark, zero-shot, DKP


## Checks
Review the printed inputs, counts, source labels and residuals. Synthetic replies validate interfaces only.

## Next steps
Use lesson 01 to distinguish already routed inflow from meteorological forcing. Calibration/validation remain native HydroCNHS workflows; no recalibration is performed here.